In [1]:
import requests
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [ ]:
class VbplCrawler:
    def __init__(self, root_path="BoPhapDienDienTu"):
        self.base_url = "http://vbpl.vn/TW/Pages/vbpq"
        self.root = root_path
        self.item_ids = self.get_ids()

    def get_ids(self):
        '''
        Get unique Item IDs across all index pages
        '''
        item_ids = set()
        demuc_path = self.root + "/demuc"

        for file in tqdm(os.listdir(demuc_path)):
            file_path = os.path.join(demuc_path, file)

            with open(file_path, 'r') as f:
                soup = BeautifulSoup(f, "html.parser")
                tags = soup.find_all('a', href=True)
                if tags:
                    for tag in tags:
                        url = tag['href']
                        if url.startswith(self.base_url):
                            try:
                                item_id = re.search(r'ItemID=(\d+)', url).group(1) # use regex to capture special cases such as ...?ItemID=139689&Keyword=41/2019/TT-BCT.html
                                item_ids.add(item_id)
                            except:
                                continue
                            
        return item_ids
    
    def len(self):
        return len(self.item_ids)
    
    def _get_and_save_html(self, session: requests.Session, url: str, output_dir:str):
        '''
        Get content of url with a requests session and write to output directory
        '''
        with open(f"{output_dir}", 'wb') as f:
            with session.get(url) as resp:
                if resp.status_code == 200: # successful requests
                    content = resp.content
                    f.write(content)
    
    def crawl_vbpl_html(self):
        '''
        Crawl the HTML of toanvan, property, history, related articles and the pdf page
        '''
        output_dirs = {
            "vbpl": f"{self.root}/vbpl",
            "history": f"{self.root}/history",
            "related": f"{self.root}/related",
            "property": f"{self.root}/property",
            "pdf": f"{self.root}/pdf"
        }

        # Ensure directories exist
        for path in output_dirs.values():
            os.makedirs(path, exist_ok=True)

        url_mappings = {
            "vbpl": "toanvan",
            "history": "lichsu",
            "related": "vanbanlienquan",
            "property": "thuoctinh",
            "pdf": "van-ban-goc"
        }

        name_mappings = {
            "vbpl": "full",
            "history": "h",
            "related": "r",
            "property": "p",
            "pdf": "pdf"
        }
        
        with requests.Session() as s:
            for item_id in tqdm(self.item_ids):
                for key, replacement in url_mappings.items():
                    # Example url: https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=148955
                    url = f"{self.base_url}-{url_mappings[key]}.aspx?ItemID={item_id}"
                    new_url = url.replace("toanvan", replacement)
                    filepath = f"{output_dirs[key]}/{name_mappings[key]}_{item_id}.html"
                    self._get_and_save_html(s, new_url, filepath)
    
    def crawl_vbpl_text(self):
        '''
        Optinal method to crawl text of full documents
        '''
        with requests.Session() as s:
            for item_id in self.item_ids:
                full_url = f"{self.base_url}-toanvan.aspx?ItemID={item_id}"
                resp = s.get(full_url)
                if resp.status_code == 200:
                    soup = BeautifulSoup(resp.text, "html.parser")
                    content_div = soup.find("div", class_="fulltext")

                    if content_div:
                        content = content_div.get_text(separator="\n", strip=True)
                        with open(f'{self.root}/vbpl/text/text_{item_id}.txt', 'w', encoding='utf-8') as f:
                            f.write(content)
    

In [3]:
crawler = VbplCrawler()
crawler.crawl_vbpl_html()

  0%|          | 0/294 [00:00<?, ?it/s]

100%|██████████| 5943/5943 [2:48:49<00:00,  1.70s/it]   
